## What is an LSTM 

**Problem it solves:**
Standard neural networks treat each input independently — they have no 
memory of what came before. LSTM (Long Short-Term Memory) networks solve 
the "vanishing gradient" problem of earlier RNNs, allowing them to 
selectively remember or forget information across long sequences.

**Why used for time series:**
Financial prices are sequential — today's rate depends on yesterday's 
rate, last week's volatility, and last month's trend. LSTM maintains a 
"cell state" that carries relevant past information forward through time, 
making it the standard baseline for any sequential prediction task.

**Inputs and outputs in our case:**
Input: a sliding window of 10 days × 9 features (returns, volatility, 
spike signals, repo rate, etc.) — shape (samples, 10, 9).
Output: a single probability (0 to 1) predicting whether INR/BRL will 
be higher tomorrow than today.

**Main computational cost:**
At every timestep, an LSTM computes 4 gate operations (input, forget, 
cell, output), each involving a full matrix multiplication over the 
hidden state. This runs on EVERY timestep regardless of whether anything 
meaningful happened — it processes quiet days and crisis days with 
identical compute. For a dataset of 1,200 training rows × 10 timesteps 
× 9 features, this is manageable — but it scales poorly.

**Why it may struggle with bursty forex data:**
INR/BRL returns are heavy-tailed (confirmed Day 7): most days are near-zero 
noise, and a small number of days carry almost all the information. LSTM 
treats every timestep equally — it cannot "focus" compute on the rare 
high-information spike events. An SNN, by contrast, fires only when 
threshold is crossed, making it naturally adapted to this bursty structure.

In [1]:
import pandas as pd
import numpy as np
import os

train = pd.read_csv("../data/processed/train_features.csv",
                    index_col=0, parse_dates=True)
val   = pd.read_csv("../data/processed/val_features.csv",
                    index_col=0, parse_dates=True)

train.index.name = "date"
val.index.name = "date"

# Separate features and target
FEATURE_COLS = [c for c in train.columns if c != "target"]
TARGET_COL   = "target"

print(f"Train rows     : {len(train)}")
print(f"Val rows       : {len(val)}")
print(f"Feature columns: {len(FEATURE_COLS)}")
print(f"Features       : {FEATURE_COLS}")
print(f"Target         : '{TARGET_COL}'")

Train rows     : 1093
Val rows       : 364
Feature columns: 9
Features       : ['daily_return', 'log_return', 'rolling_mean_7d', 'rolling_std_7d', 'price_momentum_5d', 'spike_signal', 'spike_intensity', 'inter_spike_interval', 'india_repo_rate']
Target         : 'target'


#### Handle NaNs before sequencing

In [2]:
# Drop rows where ANY feature is NaN
# These come from rolling window warm-up (first ~7 rows)
# and from missing repo rate data

train_clean = train[FEATURE_COLS + [TARGET_COL]].dropna()
val_clean   = val[FEATURE_COLS + [TARGET_COL]].dropna()

dropped_train = len(train) - len(train_clean)
dropped_val   = len(val)   - len(val_clean)

print(f"Train after dropna : {len(train_clean)} rows  (dropped {dropped_train})")
print(f"Val after dropna   : {len(val_clean)} rows  (dropped {dropped_val})")

# Confirm target is still balanced
print(f"\nTrain target balance: {train_clean[TARGET_COL].mean()*100:.1f}% up-days")
print(f"Val   target balance: {val_clean[TARGET_COL].mean()*100:.1f}% up-days")

Train after dropna : 1088 rows  (dropped 5)
Val after dropna   : 364 rows  (dropped 0)

Train target balance: 35.9% up-days
Val   target balance: 39.3% up-days


#### Sequence creation function


In [3]:
def create_sequences(data: pd.DataFrame,
                     feature_cols: list,
                     target_col: str,
                     lookback: int = 10):
    """
    Convert a flat feature matrix into LSTM-ready sequences.

    Logic:
      For each row t (starting at row `lookback`):                       .'. if lookback = 10 --> - Input = days 1-10 (features) & output = target at day 11
        X[t] = features from rows (t - lookback) to (t - 1)  ← past only
        y[t] = target at row t                               ← what we predict

    This creates a sliding window of `lookback` days.
    Each sample sees 10 days of history to predict day 11's direction.

    Parameters
    ----------
    data         : DataFrame with feature columns + target column
    feature_cols : list of column names to use as features
    target_col   : name of the target column
    lookback     : number of past timesteps to include in each sample

    Returns
    -------
    X : np.ndarray of shape (n_samples, lookback, n_features) (3D) .'. Each sample --> one sliding window of past data ,n_features --> no. of columns you choose
    y : np.ndarray of shape (n_samples,)  --> 1D
    dates : list of prediction dates (for later alignment)
    """
    X_list, y_list, date_list = [], [], []

    feature_arr = data[feature_cols].values   # (rows, n_features)
    target_arr  = data[target_col].values     # (rows,)
    dates       = data.index

    for t in range(lookback, len(data)):
        window = feature_arr[t - lookback : t]   # shape: (lookback, n_features)
        label  = target_arr[t]                   # scalar: 0 or 1

        X_list.append(window)
        y_list.append(label)
        date_list.append(dates[t])

    X = np.array(X_list, dtype=np.float32)   # (n_samples, lookback, n_features)
    y = np.array(y_list, dtype=np.float32)   # (n_samples,)

    return X, y, date_list


# Apply to train and val
LOOKBACK = 10

X_train, y_train, dates_train = create_sequences(train_clean, FEATURE_COLS, TARGET_COL, LOOKBACK)
X_val,   y_val,   dates_val   = create_sequences(val_clean,   FEATURE_COLS, TARGET_COL, LOOKBACK)

print("=" * 50)
print("SEQUENCE SHAPES")
print("=" * 50)
print(f"X_train : {X_train.shape}  (samples, lookback, features)")
print(f"y_train : {y_train.shape}")
print(f"X_val   : {X_val.shape}")
print(f"y_val   : {y_val.shape}")
print()
print(f"Reading: each sample is {LOOKBACK} days × {len(FEATURE_COLS)} features")
print(f"         predicting 1 binary label (up=1 / down=0)")

SEQUENCE SHAPES
X_train : (1078, 10, 9)  (samples, lookback, features)
y_train : (1078,)
X_val   : (354, 10, 9)
y_val   : (354,)

Reading: each sample is 10 days × 9 features
         predicting 1 binary label (up=1 / down=0)


#### Verify one sequence manually

In [4]:
# Inspect sample 0 — human-readable check
print("=== MANUAL VERIFICATION: Sample 0 ===\n")
print(f"Prediction date : {dates_train[0]}")
print(f"Window covers   : {dates_train[0]} minus {LOOKBACK} trading days before it")
print(f"Label (target)  : {int(y_train[0])}  ({'price UP' if y_train[0]==1 else 'price DOWN'})")
print(f"\nX_train[0] shape: {X_train[0].shape}  (10 timesteps × {len(FEATURE_COLS)} features)")
print(f"\nFeature matrix for this window:")

window_df = pd.DataFrame(
    X_train[0],
    columns=FEATURE_COLS,
    index=[f"t-{LOOKBACK - i}" for i in range(LOOKBACK)]
)
print(window_df.round(5).to_string())

print(f"\nVerifying no future data in window:")
print(f"  Last window timestep = t-1 (yesterday)  ✅")
print(f"  Target = tomorrow's direction             ✅")

=== MANUAL VERIFICATION: Sample 0 ===

Prediction date : 2021-03-19 00:00:00
Window covers   : 2021-03-19 00:00:00 minus 10 trading days before it
Label (target)  : 0  (price DOWN)

X_train[0] shape: (10, 9)  (10 timesteps × 9 features)

Feature matrix for this window:
      daily_return  log_return  rolling_mean_7d  rolling_std_7d  price_momentum_5d  spike_signal  spike_intensity  inter_spike_interval  india_repo_rate
t-10       0.00269     0.00268        73.123238         0.14276            0.00616           0.0          0.00000                   4.0             6.35
t-9       -0.00756    -0.00758        73.073631         0.18498           -0.00487           1.0          0.00756                   5.0             6.35
t-8       -0.00096    -0.00096        73.048668         0.22319           -0.00582           0.0          0.00000                   1.0             6.35
t-7       -0.00092    -0.00092        72.978271         0.26618           -0.00674           0.0          0.00000     

#### Check for NaN or Inf in sequences

In [5]:
print("=== DATA QUALITY CHECK ===")
print(f"X_train NaN count : {np.isnan(X_train).sum()}")
print(f"X_train Inf count : {np.isinf(X_train).sum()}")
print(f"y_train NaN count : {np.isnan(y_train).sum()}")
print(f"X_val   NaN count : {np.isnan(X_val).sum()}")
print(f"X_val   Inf count : {np.isinf(X_val).sum()}")

if np.isnan(X_train).sum() == 0 and np.isinf(X_train).sum() == 0:
    print("\n✅ Sequences are clean — ready for model input")
else:
    print("\n⚠️  NaN/Inf detected — check feature engineering step")

=== DATA QUALITY CHECK ===
X_train NaN count : 0
X_train Inf count : 0
y_train NaN count : 0
X_val   NaN count : 0
X_val   Inf count : 0

✅ Sequences are clean — ready for model input


#### Install and verify TensorFlow / Keras

In [6]:
# Run this cell — if it fails, run the pip install below first
# Terminal: pip install tensorflow

try:
    import tensorflow as tf
    from tensorflow import keras
    print(f"TensorFlow version : {tf.__version__}")
    print(f"Keras version      : {keras.__version__}")
    print(f"GPU available      : {len(tf.config.list_physical_devices('GPU')) > 0}")
    print("✅ TensorFlow/Keras ready")
    FRAMEWORK = "tensorflow"
except ImportError:
    print("TensorFlow not installed.")
    print("Run in terminal: pip install tensorflow")
    print("\nAlternative: if you prefer PyTorch, run: pip install torch")
    print("We'll add PyTorch LSTM option in Day 18.")
    FRAMEWORK = "not_installed"


TensorFlow version : 2.21.0
Keras version      : 3.12.1
GPU available      : False
✅ TensorFlow/Keras ready


#### LSTM Architecture Plan

### Architecture
```
Input shape: (batch, 10, 9)      ← 10 days lookback, 9 features
      │
      ▼
┌─────────────────────────────┐
│  LSTM(64 units)             │  ← learns temporal patterns across 10 days
│  return_sequences=False     │  ← only output at final timestep
└─────────────────────────────┘
      │
      ▼
┌─────────────────────────────┐
│  Dropout(0.2)               │  ← prevents overfitting on small dataset
└─────────────────────────────┘
      │
      ▼
┌─────────────────────────────┐
│  Dense(1, sigmoid)          │  ← outputs P(price goes up tomorrow)
└─────────────────────────────┘
      │
      ▼
Output: scalar in [0, 1]         ← threshold at 0.5 → binary prediction

Loss    : binary_crossentropy
Optimizer: Adam(lr=0.001)
Metric  : accuracy + AUC
Epochs  : 50 (with early stopping on val loss, patience=10)
```

### Baseline statement
"My LSTM baseline predicts next-day INR/BRL price direction (up or down)
using 10 days of history across 9 features. It will be trained on 
~720 sequences and evaluated on ~240 validation sequences. Its accuracy
and AUC on the validation set set the benchmark that the SNN must beat."

### Why this architecture and not something deeper?
- Dataset has ~720 training sequences — too small for deep stacking
- Two LSTM layers with a small dataset leads to overfitting
- Dropout(0.2) is conservative — adds regularisation without killing signal
- This is intentionally simple: the baseline should be honest, not heroic

#### Summary printout

In [7]:
print("=" * 60)
print("DAY 17 SUMMARY — LSTM BASELINE PREPARATION")
print("=" * 60)
print(f"\nBaseline statement:")
print(f"  My LSTM will predict next-day INR/BRL direction (up/down)")
print(f"  using {LOOKBACK} days of history × {len(FEATURE_COLS)} features.")
print(f"\nSequences ready:")
print(f"  X_train : {X_train.shape}  |  y_train : {y_train.shape}")
print(f"  X_val   : {X_val.shape}    |  y_val   : {y_val.shape}")
print(f"\nArchitecture (Day 18):")
print(f"  LSTM(64) → Dropout(0.2) → Dense(1, sigmoid)")
print(f"  Loss: binary_crossentropy | Optimizer: Adam")
print(f"\nFramework installed: {FRAMEWORK}")
print(f"\nNext step (Day 18): Train this LSTM and record val accuracy + AUC.")
print("=" * 60)

DAY 17 SUMMARY — LSTM BASELINE PREPARATION

Baseline statement:
  My LSTM will predict next-day INR/BRL direction (up/down)
  using 10 days of history × 9 features.

Sequences ready:
  X_train : (1078, 10, 9)  |  y_train : (1078,)
  X_val   : (354, 10, 9)    |  y_val   : (354,)

Architecture (Day 18):
  LSTM(64) → Dropout(0.2) → Dense(1, sigmoid)
  Loss: binary_crossentropy | Optimizer: Adam

Framework installed: tensorflow

Next step (Day 18): Train this LSTM and record val accuracy + AUC.
